# Lab3 CPA

本实验用于演示：
1. 显示曲线文件。
2. 使用CrackNuts的Squirrel库进行CPA分析破解密钥。

## 准备工作

软件：
1. 安装最新[`CrackNuts`](https://cracknuts.cn/docs/doc/getting-started/host_sdk_installation)控制分析软件。
2. 安装`cracknuts-squirrel`
   ```shell
   pip install git+https://github.com/cracknuts-team/cracknuts-squirrel.git
   ```

数据：  
波形曲线文件`20250521110621(aes).zarr`、`20250717141034(aes_protocol_unaligned).zarr`可在 [https://pan.baidu.com/s/1PXyKqeTfemepZ-wD9gDwYQ?pwd=2cda](https://pan.baidu.com/s/1PXyKqeTfemepZ-wD9gDwYQ?pwd=2cda) 下载，其放置在 `CrackNuts/Tutorials/traces` 文件夹下，下载后后续曲线文件的路径需要修改为你的曲线路径。

## 实验步骤

1. 展示波形文件
2. CPA分析破解密钥

In [ ]:
from cracknuts_squirrel.cpa_analysis import CPAAnalysis
import cracknuts as cn
import zarr 
from cracknuts.trace import ScarrTraceDataset

In [ ]:
trace_path = r'../traces/20250521110621(aes).zarr'

# 加载 scarr 格式的数据集
ds = ScarrTraceDataset.load(trace_path)

In [ ]:
# Print dataset info
ds.info()

展示波形

In [ ]:
# Show trace panel
pt = cn.panel_trace()
pt.set_trace_dataset(ds)
pt

波形对齐

有时，采集到的波形，各个曲线之间可能由于触发导致时钟并不一致，此时可以使用squirrel的波形对齐功能对其进行修正。

In [ ]:
dataset_path = r'../traces/20250717141034(aes_protocol_unaligned).zarr' # 未对齐的曲线

# 加载 scarr 格式的数据集
ds = ScarrTraceDataset.load(dataset_path)
align_pt = cn.panel_trace()
align_pt.set_trace_dataset(ds)
align_pt.change_range(4000, 5000)  # 缩放，展示曲线细节
align_pt

观察以上波形可以看到，波形时间轴混乱

In [ ]:
# 调用 squirrel 开始对齐波形

from cracknuts_squirrel.staticalgin import Staticalign

aligner = Staticalign(input_path=dataset_path)
aligner.auto_out_filename()
aligner.set_ref(ref_range=(16000, 17000))  # 设置参考曲线范围(选择曲线中特征比较明显且不重复的部分)
aligner.align_curves(method='correlation')  # 使用相关性方法对齐曲线

In [ ]:
# 再次查看曲线（对齐后的）
dataset_path = r'../traces/20250717141034(aes_protocol_unaligned)_Staticalign.zarr' # 未对齐的曲线(该文件是对齐后自动生成的结果)

# 加载 scarr 格式的数据集
ds = ScarrTraceDataset.load(dataset_path)
align_pt = cn.panel_trace()
align_pt.set_trace_dataset(ds)
align_pt.change_range(4000, 5000)  # 缩放，展示曲线细节
align_pt


开始分析数据

In [ ]:
dataset_path = r"../traces/20250521110621(aes).zarr"

cpa = CPAAnalysis(input_path=dataset_path)
cpa.auto_out_filename()
cpa.perform_cpa()

In [ ]:
# 上面执行完成后，密钥就已经分析出来，我们可以通过相关性矩阵直接 检索出每个自己的密钥

import zarr
import numpy as np

# 这里是squirrel分析后的相关性系数矩阵，我们可以操作这类来获取密钥等信息
correlation_zarr = zarr.open(r'../traces/20250521110621(aes)_CPAAnalysis.zarr')
correlation = correlation_zarr["/0/0/correlation"]

# 第一步：提取每个字节每个密钥猜测对应的最大相关性
# 得到 shape = (256, 16)
max_corr_per_guess = np.max(np.abs(correlation), axis=2)

# 第二步：在每个字节上找出相关性最大的密钥猜测
# 取 axis=0（对256个猜测做 argmax），结果是每个字节一个最优 key
# shape = (16,)
best_key_per_byte = np.argmax(max_corr_per_guess, axis=0)

# 输出最终密钥（16 字节）
final_key = best_key_per_byte.tolist()
hex_key = ' '.join(f'{b:02x}' for b in final_key)
print("密钥:", hex_key)


In [ ]:
# 我们还可以观察其中一个密钥字节的相关性系数排名，例如第0字节
correlation_byte0 = correlation[:,0,:]
correlation_byte0.shape

max_idx_per_correlation = np.argmax(np.abs(correlation_byte0), axis=1)
max_value_per_correlation = correlation_byte0[range(256), max_idx_per_correlation]

top_10_idx = np.argsort(max_value_per_correlation)[::-1][:10]

for rank, idx in enumerate(top_10_idx):
    print(f"第 {rank} 候选值: 0x{idx:0X}，对应的相关系数为: {max_value_per_correlation[idx]}")


In [ ]:
# 这里定义一个函数，用于显示相关性曲线

import numpy as np
import matplotlib.pyplot as plt

def plot_correlation_peaks(correlation_data, bytes_index, the_key):
    
    x = np.arange(0, 5000)
    
    fig, ax = plt.subplots(figsize=(30, 4))
    
    for i in range(256):
        if i == the_key:
            continue
        ax.plot(x, correlation_data[i, bytes_index, :5000], color='gray', linewidth=0.5, alpha=0.3)
            
    ax.plot(x, correlation_data[the_key, bytes_index, :5000], color='red', linewidth=1.0)
    
    ax.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    
    plt.show()

In [ ]:
plot_correlation_peaks(correlation, 0, 0x11)  # ,这里的 0x11 是通过上面分析出的密钥的得到的

In [ ]:
plot_correlation_peaks(correlation, 1, 0x22)

In [ ]:
# 绘制所有密钥的相关性曲线

import numpy as np
import matplotlib.pyplot as plt

colors = [
    # Group 1: Red to Orange
    "#e41a1c", "#ff5733", "#ff7f0e", "#ff9900",

    # Group 2: Blue to Teal
    "#1f77b4", "#3399cc", "#40b4d4", "#20c997",

    # Group 3: Green to Yellow-Green
    "#2ca02c", "#66bb6a", "#a1d99b", "#d4e157",

    # Group 4: Purple to Pink
    "#984ea3", "#ba68c8", "#e377c2", "#ff69b4"
]

linestyle_list = ['-', '--']


x = np.arange(0, 2000)

fig, ax = plt.subplots(figsize=(30, 4))

ax.plot(x, correlation[0x11, 0, :2000].T, linewidth=1.0, color=colors[0], linestyle=linestyle_list[0], label='0x11')
ax.plot(x, correlation[0x22, 1, :2000].T, linewidth=1.0, color=colors[1], linestyle=linestyle_list[1], label='0x22')
ax.plot(x, correlation[0x33, 2, :2000].T, linewidth=1.0, color=colors[2], linestyle=linestyle_list[0], label='0x33')
ax.plot(x, correlation[0x44, 3, :2000].T, linewidth=1.0, color=colors[3], linestyle=linestyle_list[1], label='0x44')
ax.plot(x, correlation[0x55, 4, :2000].T, linewidth=1.0, color=colors[4], linestyle=linestyle_list[0], label='0x55')
ax.plot(x, correlation[0x66, 5, :2000].T, linewidth=1.0, color=colors[5], linestyle=linestyle_list[1], label='0x66')
ax.plot(x, correlation[0x77, 6, :2000].T, linewidth=1.0, color=colors[6], linestyle=linestyle_list[0], label='0x77')
ax.plot(x, correlation[0x88, 7, :2000].T, linewidth=1.0, color=colors[7], linestyle=linestyle_list[1], label='0x88')
ax.plot(x, correlation[0x99, 8, :2000].T, linewidth=1.0, color=colors[8], linestyle=linestyle_list[0], label='0x99')
ax.plot(x, correlation[0x00, 9, :2000].T, linewidth=1.0, color=colors[9], linestyle=linestyle_list[1], label='0x00')
ax.plot(x, correlation[0xaa, 10, :2000].T, linewidth=1.0, color=colors[10], linestyle=linestyle_list[0], label='0xaa')
ax.plot(x, correlation[0xbb, 11, :2000].T, linewidth=1.0, color=colors[11], linestyle=linestyle_list[1], label='0xbb')
ax.plot(x, correlation[0xcc, 12, :2000].T, linewidth=1.0, color=colors[12], linestyle=linestyle_list[0], label='0xcc')
ax.plot(x, correlation[0xdd, 13, :2000].T, linewidth=1.0, color=colors[13], linestyle=linestyle_list[1], label='0xdd')
ax.plot(x, correlation[0xee, 14, :2000].T, linewidth=1.0, color=colors[14], linestyle=linestyle_list[0], label='0xee')
ax.plot(x, correlation[0xff, 15, :2000].T, linewidth=1.0, color=colors[15], linestyle=linestyle_list[1], label='0xff')

ax.grid(True, linestyle='--', alpha=0.3)
ax.legend(loc='upper right', fontsize='small', ncol=2)

plt.tight_layout()

# 显示图像
plt.show()